<a href="https://colab.research.google.com/github/lilyzhengliwang/Problem-Set-2/blob/main/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [13]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""

    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    log_eng = np.log(engagement_minutes)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()

    logit_p = np.log(0.20 / 0.80) - 0.25 * engagement_z
    had_support_ticket = rng.binomial(1, 0.22, n)

    s = (base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, n))

    satisfaction = np.clip(np.rint(s), 1, 5).astype(int)

    return pd.DataFrame({
       "engagement_minutes" : engagement_minutes,
        "engagement_z" : engagement_z,
        "had_support_ticket" : had_support_ticket,
        "s" : s,
        "satisfaction" : satisfaction,
    })


population_before = simulate_population(BASE_BEFORE)
population_after = simulate_population(BASE_AFTER)

true_before, true_after = population_before.satisfaction.mean(), population_after.satisfaction.mean()

print(f"True mean before: {true_before:.3f}; after: {true_after:.3f}; lift: {true_after - true_before:.3f}")
(f"Ticket rate before/after: {population_before.had_support_ticket.mean():.3f} / {population_after.had_support_ticket.mean():.3f}")
print(f"correlation (engagement_z, ticket): {population_after[['engagement_z','had_support_ticket']].corr().iloc[0,1]:.3f}")
print(population_after.groupby('had_support_ticket').satisfaction.mean())

True mean before: 3.290; after: 3.386; lift: 0.096
correlation (engagement_z, ticket): 0.003
had_support_ticket
0    3.517
1    2.927
Name: satisfaction, dtype: float64


*Your answer here.*

In the simulated population the true mean before redesign was 3.31, and the true mean afterwards was 3.40, with a lift of roughly 0.09 points. Users who filed support tickets averaged 2.92 while users who didn't averaged 3.52, and there seems to be a negative correlation between engagement level of users and tickets filed. Both engagement level and support ticket epxerience can affect how satisfied people are.

### Step 1.2: Impose the response mechanism (15 pts)

In [14]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""

    score = (a + b*(df.s - 3.3) + 0.30*df.engagement_z - 0.25*df.had_support_ticket)
    p = 1 / (1 + np.exp(-score))
    return rng.random(len(df)) < p


population_before['responded'] = respond(population_before, a=-1.0, b=0.45)
population_after['responded'] = respond(population_after, a = -2.75, b=2.00)

for name, d in [("before", population_before), ("after", population_after)]:
    print(f"{name}: true {d.satisfaction.mean():.3f}  observed {d.satisfaction[d.responded].mean():.3f}  "
          f"response rate {d.responded.mean():.3f}")


before: true 3.290  observed 3.600  response rate 0.275
after: true 3.386  observed 4.304  response rate 0.152


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

We imposed a response mechanism on the simulated users from the previous step, where a sets the overall response rate and b sets the selection bias. Before the redesign, respondents were around 0.3 points higher than the true mean, and afterwards they were about 0.9 points higher, suggesting that some of the improvement comes from a change in who decided to answer and not necesarily how satisfied users really are; fewer people answered but were more satisfied, with the response rate dropping from ~27% to ~15%. The failure mode is selection bias, and a larger sample does not fix because it only increases the precision but not the accuracy; it will cause the standard error to decrease, but the estimated mean will still be away from the true mean.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [15]:
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Your own resampling loop; returns the B bootstrap statistics."""
    x = np.asarray(x)
    n = len(x)
    return np.array([stat(rng.choice(x, size=n, replace=True)) for _ in range(B)])

response_before = population_before.satisfaction[population_before.responded].to_numpy()
response_after  = population_after.satisfaction[population_after.responded].to_numpy()

boot_before = bootstrap_by_hand(response_before, np.mean)
boot_after  = bootstrap_by_hand(response_after,  np.mean)

lo_b, hi_b = np.percentile(boot_before, [2.5, 97.5])
lo_a, hi_a = np.percentile(boot_after,  [2.5, 97.5])


res_b = stats.bootstrap((response_before,), np.mean, n_resamples=2_000, method="percentile", random_state=1)
print(f"before: by hand [{lo_b:.3f}, {hi_b:.3f}] | scipy [{res_b.confidence_interval.low:.3f}, {res_b.confidence_interval.high:.3f}]")
print(f"after:  by hand [{lo_a:.3f}, {hi_a:.3f}]")

before: by hand [3.585, 3.615] | scipy [3.585, 3.614]
after:  by hand [4.289, 4.319]


*Your answer here.*
The intervals are narrow, having a range of about 0.03 plus-minus points, since many people answered each time. The "before" interval is [3.593, 3.622], and the "after" interval is [4.285, 4.315]. It seems that both intervals do not contain the true mean, showing some bias - the intervals are precise but not accurate.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [16]:
def coverage(pop, a=None, b=None, selected=True, n_reps=400):

    true_mean = pop.satisfaction.mean()
    n = int(round(pop.responded.mean() * len(pop)))
    y = pop.satisfaction.to_numpy()
    n_covered = 0

    for _ in range(n_reps):
        if selected:
            take = respond(pop, a, b, rng=rng)
            s = y[take]

        else:
            s = rng.choice(y, size=n, replace=False)

        se = s.std(ddof=1) / np.sqrt(len(s))
        n_covered += (s.mean() - 1.96 * se) <= true_mean <= (s.mean() + 1.96 * se)

    cov = n_covered / n_reps
    return cov, np.sqrt(cov * (1 - cov) / n_reps)


a_before, b_before = -1.0, 0.45
a_after, b_after = -2.75, 2.00

rows = []
for name, pop, a, b in [("before", population_before, a_before, b_before),
                        ("after", population_after, a_after, b_after)]:

    for label, selection in [("random sample", False), ("self-selected survey", True)]:
        cov, mcse = coverage(pop, a, b, selected = selection)
        rows.append((name, label, cov, mcse))



print(pd.DataFrame(rows, columns=["survey period", "who answers", "coverage", "Monte Carlo SE"]))


  survey period           who answers  coverage  Monte Carlo SE
0        before         random sample     0.985           0.006
1        before  self-selected survey     0.000           0.000
2         after         random sample     0.968           0.009
3         after  self-selected survey     0.000           0.000


*Your answer here.*

When drawing a random sample with the same number of respondents, covrage is above 95%; respondents are a large share of all users. None of the 400 repeated surveys produced an interval containing the true mean in either survey period, and repeating the survey helps to reduce the SE but does not help with bias.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [17]:
# survey respondents only
x_before = population_before.satisfaction[population_before.responded].to_numpy()
x_after = population_after.satisfaction[population_after.responded].to_numpy()

# Welch's t-test
res = stats.ttest_ind(x_after, x_before, equal_var=False)

diff = x_after.mean() - x_before.mean()
se = np.sqrt(x_before.var(ddof=1)/len(x_before) + x_after.var(ddof=1)/len(x_after))
pooled_sd = np.sqrt(((len(x_before)-1)*x_before.var(ddof=1) + (len(x_after)-1)*x_after.var(ddof=1))
                    / (len(x_before) + len(x_after) - 2))
cohens_d = diff / pooled_sd


print(f"n: {len(x_before):,} before, {len(x_after):,} after")
print(f"means: {x_before.mean():.3f} -> {x_after.mean():.3f}")
print(f"difference: {diff:.3f}  95% CI [{diff - 1.96*se:.3f}, {diff + 1.96*se:.3f}]")
print(f"Welch t = {res.statistic:.1f}, p = {res.pvalue:.2e}")
print(f"Cohen's d = {cohens_d:.2f}")


n: 13,768 before, 7,597 after
means: 3.600 -> 4.304
difference: 0.704  95% CI [0.683, 0.725]
Welch t = 65.2, p = 0.00e+00
Cohen's d = 0.86


*Your answer here.*

A Welch two-sample t-test on the survey respondents gives a differene of about 0.73 points with a 95% confidence interval of [0.702, 0.750], t of about 60, and a p-value of zero. The effect is large, as shown by Cohen's d of 0.80, meaning that the test ran correctly. However, statistical significance is not the issue here because the test cannot fix bias and cannot see who doesn't answer. It treats each period's respondents as a random sample, but they are actually not random; intervals that claimed 95% coverage had 0% coverage of the true mean.

### Step 3.2: Power, before the fact

In [18]:
from statsmodels.stats.power import TTestIndPower
power = TTestIndPower()

true_diff = population_after.satisfaction.mean() - population_before.satisfaction.mean()   # ~0.10
sd = np.sqrt((population_before.satisfaction.var(ddof=1) + population_after.satisfaction.var(ddof=1)) / 2)

# sample size for a true improvement of 0.1 points
d_planned = 0.10 / sd
n_per_group = power.solve_power(effect_size=d_planned, alpha=0.05, power=0.80)
print(f"d = {d_planned:.3f} (0.10 points / sd {sd:.2f}) -> {n_per_group:,.0f} per group")

# smallest effect detectable at the sample sizes actually obtained
n_after  = population_after.responded.sum()
n_before = population_before.responded.sum()
d_min = power.solve_power(effect_size=None, nobs1=n_after, ratio=n_before / n_after, alpha=0.05, power=0.80)
print(f"with {n_after:,} vs {n_before:,} respondents: d = {d_min:.3f} = {d_min * sd:.3f} points")

d = 0.113 (0.10 points / sd 0.89) -> 1,236 per group
with 7,597 vs 13,768 respondents: d = 0.040 = 0.036 points


*Your answer here.*
For a true improvement of 0.1 points, around 1,236 per sample gives 80% power, or total of 2,472. The smallest detectable effect at the actual sample sizes, with 7,597 respondents after and 13,768 before, you can detect an effect size of Cohen's d = 0.04, or an equivalent of a 0.036 point improvement, which is much smaller than the improvement of 0.1 points we are looking at.

In [19]:
# does larger sample size make the estimate better?
def simulate_followup(k, selected, reps=300):
    """Invite k users per period. selected=True: they choose to answer (the real mechanism).
    selected=False: a random sample with the same expected number of respondents."""
    est, covered = [], 0
    for _ in range(reps):
        groups = []
        for pop, a, b in [(population_before, a_before, b_before), (population_after, a_after, b_after)]:
            if selected:
                sub = pop.iloc[rng.choice(len(pop), k, replace=False)]
                groups.append(sub.satisfaction.to_numpy()[respond(sub, a, b, rng=rng)])
            else:
                m = int(k * pop.responded.mean())
                groups.append(rng.choice(pop.satisfaction.to_numpy(), m, replace=False))
        xb, xa = groups
        diff = xa.mean() - xb.mean()
        se = np.sqrt(xa.var(ddof=1)/len(xa) + xb.var(ddof=1)/len(xb))
        est.append(diff)
        covered += (diff - 1.96*se <= true_diff <= diff + 1.96*se)
    est = np.array(est)
    return est.mean(), est.mean() - true_diff, est.std(), np.sqrt(((est - true_diff)**2).mean()), covered / reps

rows = []
for k in [2_000, 5_000, 10_000, 25_000, 50_000]:
    for sel in [False, True]:
        rows.append((k, "self-selected" if sel else "random", *simulate_followup(k, sel)))
print(pd.DataFrame(rows, columns=["invited", "who answers", "mean estimate", "bias", "sd", "rmse", "CI coverage"]).round(3))

   invited    who answers  mean estimate   bias     sd   rmse  CI coverage
0     2000         random          0.107  0.011  0.064  0.065        0.943
1     2000  self-selected          0.707  0.611  0.052  0.613        0.000
2     5000         random          0.095 -0.001  0.038  0.038        0.967
3     5000  self-selected          0.699  0.604  0.035  0.605        0.000
4    10000         random          0.095 -0.000  0.028  0.028        0.960
5    10000  self-selected          0.701  0.606  0.024  0.606        0.000
6    25000         random          0.096 -0.000  0.017  0.017        0.953
7    25000  self-selected          0.701  0.605  0.013  0.605        0.000
8    50000         random          0.095 -0.001  0.011  0.011        0.973
9    50000  self-selected          0.702  0.606  0.009  0.606        0.000


No, a larger sample size does not make the estimate better. I repeated a followup experiment testing different sample sizes, ranging from 2,000 per period to 50,000 per period. In the random sample, the mean is close to the truth and the error falls, with coverage above 95%. However, the results from self selected samples continue to hover around the wrong mean estimate at around 0.70, meaning that the bias is around 0.60 points; coverage remained at 0. This shows that increasing sample size reduces noise but doesn't reduce bias. A larger sample only gives a more precise estimate of the wrong mean estimate, so more effort should be put into random sampling rather than gathering larger sample sizes for the survey.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

###AI Transcript: Gemini 3, Oct 26

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

### AI Response V1
EvaluationNo. Given your mechanism—where the probability of responding depends directly on satisfaction itself—this correction will not recover the true population mean.Mechanism Check & Assumptions1. The Key Assumption: Missing At Random (MAR)Inverse-Probability Weighting (IPW) relies on the Missing At Random (MAR) assumption (also known as conditional independence or unconfoundedness). Mathematically, this assumes that the probability of response ($R_i = 1$) is independent of the survey outcome ($Y_i$, i.e., satisfaction), conditional on the observed covariates ($X_i$):$$P(R_i = 1 \mid Y_i, X_i) = P(R_i = 1 \mid X_i)$$Under MAR, once we control for engagement_minutes and had_support_ticket, remaining non-response is essentially random with respect to satisfaction.2. Violation: Missing Not At Random (MNAR)If the true response mechanism depends on satisfaction itself (e.g., highly satisfied or deeply dissatisfied users are more likely to respond than neutral users) even after accounting for engagement and support tickets, the data is Missing Not At Random (MNAR).Because your logistic model only includes engagement_minutes and had_support_ticket as predictors, it cannot capture the variation in response propensity driven by satisfaction. Consequently, the estimated response probabilities ($\hat{p}_i$) will be misspecified, the resulting weights will fail to adjust for the satisfaction-driven non-response bias, and the weighted mean will remain biased.Python Implementation (IPW & Bootstrap)Here is the implementation using pandas, numpy, and scikit-learn that fits the logistic response model, calculates IPW weights, and computes a nonparametric bootstrap confidence interval.Pythonimport numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


def compute_ipw_estimate(df, n_boot=1000, random_state=42):
  """Computes the IPW-corrected mean of satisfaction and its bootstrap CI.

  Parameters:
  - df: DataFrame containing 'satisfaction', 'engagement_minutes',
        'had_support_ticket', and 'responded' (1 if responded, 0 if not).
  - n_boot: Number of bootstrap iterations.
  """
  # 1. Fit logistic model on original data
  X = df[["engagement_minutes", "had_support_ticket"]]
  y = df["responded"]

  model = LogisticRegression(random_state=random_state)
  model.fit(X, y)

  # Predict response probabilities for respondents
  df_orig = df.copy()
  df_orig["p_hat"] = model.predict_proba(X)[:, 1]

  # Compute point estimate of weighted mean
  respondents = df_orig[df_orig["responded"] == 1]
  weights = 1.0 / respondents["p_hat"]
  point_estimate = np.sum(respondents["satisfaction"] * weights) / np.sum(weights)

  # 2. Bootstrap for Confidence Interval
  boot_estimates = []
  rng = np.random.default_rng(random_state)

  for _ in range(n_boot):
    # Resample entire population dataset with replacement
    boot_sample = df.sample(n=len(df), replace=True, random_state=rng.integers(0, 1e9))

    X_boot = boot_sample[["engagement_minutes", "had_support_ticket"]]
    y_boot = boot_sample["responded"]

    try:
      # Re-fit model on bootstrap sample (capturing sampling variance of weights)
      boot_model = LogisticRegression()
      boot_model.fit(X_boot, y_boot)

      boot_sample["p_hat"] = boot_model.predict_proba(X_boot)[:, 1]
      boot_resp = boot_sample[boot_sample["responded"] == 1]

      if len(boot_resp) == 0:
        continue

      boot_weights = 1.0 / boot_resp["p_hat"]
      boot_mean = np.sum(boot_resp["satisfaction"] * boot_weights) / np.sum(
          boot_weights
      )

      if not np.isnan(boot_mean):
        boot_estimates.append(boot_mean)
    except Exception:
      # Skip degenerate bootstrap iterations
      continue

  # 3. Percentile Confidence Interval
  ci_lower = np.percentile(boot_estimates, 2.5)
  ci_upper = np.percentile(boot_estimates, 97.5)

  return point_estimate, (ci_lower, ci_upper)


# Example usage format:
# df = pd.DataFrame({
#     'satisfaction': [...],          # NaN for non-respondents
#     'engagement_minutes': [...],
#     'had_support_ticket': [...],
#     'responded': [...]              # 1 or 0
# })
# mean_est, ci = compute_ipw_estimate(df)



### Prompt V2
My 1–3 outputs produced the following (3.290 / 3.386 true, 3.600 / 4.304 observed). you can correct more than one period at a time and compare against the known truth. The PM’s claim is about the lift. The narrow interval also looks like a result when it misses the true mean. Try to correct both periods in one pass and reports the corrected lift with a bootstrap interval. print out weight diagnostics (max weight, effective n) as well

###AI Response V2
The results you are observing illustrate the exact danger of applying Inverse-Probability Weighting (IPW) under Missing Not At Random (MNAR) conditions.Diagnosis of Your ResultsOverestimation of Satisfaction (MNAR Bias):Period 1 True: $3.290$ $\rightarrow$ Observed: $3.600$Period 2 True: $3.386$ $\rightarrow$ Observed: $4.304$True Lift: $+0.096$ points ($\approx +2.9\%$)Observed Lift: $+0.704$ points ($\approx +19.6\%$)Why this happens: Because higher satisfaction directly increases response probability, happy users self-select into responding at disproportionate rates, severely inflating both individual period means and the observed lift.False Precision (Narrow Confidence Intervals):Under MNAR, IPW fits a response model on observables (engagement_minutes, had_support_ticket) and assigns re-weighting values.Standard bootstrap resampling measures sampling variability, not model bias. If the model itself is structurally misspecified (omitting unobserved outcome-driven selection), the bootstrap will yield a tight, precise interval around a wrong, biased estimate.
Python Code: Multi-Period Correction, Diagnostics & Lift BootstrapThis modular script fits the propensity model, calculates weight diagnostics (Max Weight and Kish's Effective Sample Size $n_{\text{eff}}$), compares true vs. IPW estimates for both periods, and calculates the corrected lift along with a non-parametric bootstrap interval.
from sklearn.linear_model import LogisticRegression

COVS = ["engagement_z", "had_support_ticket"]     # what an analyst could actually observe about every user

def ipw_mean(df):
    """Fit P(responded | COVS), weight each respondent by 1/p_hat, return the weighted mean and the weights."""
    X = df[COVS].to_numpy()
    r = df["responded"].to_numpy().astype(bool)
    p_hat = LogisticRegression().fit(X, r).predict_proba(X)[:, 1]
    w = 1.0 / p_hat[r]                              # rarely-responding types count for more
    y = df["satisfaction"].to_numpy()[r]
    return np.sum(w * y) / np.sum(w), w             # weighted mean of the respondents

def ipw_bootstrap(df, B=300, rng=rng):
    """Resample ALL users (responders and not), refit the response model, recompute the weighted mean."""
    n = len(df)
    return np.array([ipw_mean(df.iloc[rng.integers(0, n, n)])[0] for _ in range(B)])

est, boot, w_used = {}, {}, {}
for name, pop in [("before", population_before), ("after", population_after)]:
    est[name], w_used[name] = ipw_mean(pop)
    boot[name] = ipw_bootstrap(pop)

true = {"before": true_before, "after": true_after}
naive = {"before": x_before.mean(), "after": x_after.mean()}
rows = []
for k in ["before", "after"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    rows.append((k, true[k], naive[k], est[k], est[k] - true[k], f"[{lo:.3f}, {hi:.3f}]",
                 w_used[k].max(), (w_used[k].sum() ** 2) / (w_used[k] ** 2).sum()))

lift_boot = boot["after"] - boot["before"]
lo, hi = np.percentile(lift_boot, [2.5, 97.5])
rows.append(("lift (after - before)", true_after - true_before, naive["after"] - naive["before"],
             est["after"] - est["before"], (est["after"] - est["before"]) - (true_after - true_before),
             f"[{lo:.3f}, {hi:.3f}]", np.nan, np.nan))

print(pd.DataFrame(rows, columns=["period", "true mean", "naive (respondents)", "IPW mean", "IPW minus true",
                                  "95% bootstrap CI", "max weight", "effective n"]).round(3).to_string(index=False))

Key Takeaways for Product & PM ClaimsModel Assumptions Fail Under MNAR: If satisfaction itself drives response, weighting on $X = \{\text{engagement\_minutes}, \text{had\_support\_ticket}\}$ reduces bias only to the extent that $X$ correlates with satisfaction. It cannot eliminate the direct bias.Overconfident PM Claims: If the PM claims a large lift (e.g., $+0.704$ observed vs. $+0.096$ true), IPW will likely still show a heavily inflated lift. The narrow confidence interval confirms that the estimate is statistically precise, but structurally inaccurate.Remediation Strategy: When non-response depends on unobservables, consider sensitivity analysis (e.g., Manski bounds or pattern-mixture models) or incorporate instrumental variables / proxy variables directly linked to satisfaction.

In [21]:
## Version 2

from sklearn.linear_model import LogisticRegression

COVS = ["engagement_z", "had_support_ticket"]     # what an analyst could actually observe about every user

def ipw_mean(df):
    """Fit P(responded | COVS), weight each respondent by 1/p_hat, return the weighted mean and the weights."""
    X = df[COVS].to_numpy()
    r = df["responded"].to_numpy().astype(bool)
    p_hat = LogisticRegression().fit(X, r).predict_proba(X)[:, 1]
    w = 1.0 / p_hat[r]                              # rarely-responding types count for more
    y = df["satisfaction"].to_numpy()[r]
    return np.sum(w * y) / np.sum(w), w             # weighted mean of the respondents

def ipw_bootstrap(df, B=300, rng=rng):
    """Resample ALL users (responders and not), refit the response model, recompute the weighted mean."""
    n = len(df)
    return np.array([ipw_mean(df.iloc[rng.integers(0, n, n)])[0] for _ in range(B)])

est, boot, w_used = {}, {}, {}
for name, pop in [("before", population_before), ("after", population_after)]:
    est[name], w_used[name] = ipw_mean(pop)
    boot[name] = ipw_bootstrap(pop)

true = {"before": true_before, "after": true_after}
naive = {"before": x_before.mean(), "after": x_after.mean()}
rows = []
for k in ["before", "after"]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    rows.append((k, true[k], naive[k], est[k], est[k] - true[k], f"[{lo:.3f}, {hi:.3f}]",
                 w_used[k].max(), (w_used[k].sum() ** 2) / (w_used[k] ** 2).sum()))

lift_boot = boot["after"] - boot["before"]
lo, hi = np.percentile(lift_boot, [2.5, 97.5])
rows.append(("lift (after - before)", true_after - true_before, naive["after"] - naive["before"],
             est["after"] - est["before"], (est["after"] - est["before"]) - (true_after - true_before),
             f"[{lo:.3f}, {hi:.3f}]", np.nan, np.nan))

print(pd.DataFrame(rows, columns=["period", "true mean", "naive (respondents)", "IPW mean", "IPW minus true",
                                  "95% bootstrap CI", "max weight", "effective n"]).round(3).to_string(index=False))

               period  true mean  naive (respondents)  IPW mean  IPW minus true 95% bootstrap CI  max weight  effective n
               before      3.290                3.600     3.464           0.174   [3.449, 3.478]       8.124    12406.357
                after      3.386                4.304     4.143           0.757   [4.123, 4.162]      44.005     4785.136
lift (after - before)      0.096                0.704     0.680           0.584   [0.651, 0.705]         NaN          NaN


1. **Your revision:** what you changed and why.
Version 1 corrected one period's mean in isolation and never compared it to the known truth, but the PM's claim is about the lift between periods. V2 corrects both periods with the same function and added true means.
2. **Your check:** the corrected mean against your known true mean, with both numbers.
Before the redesign the corrected mean is 3.464 against a true mean of 3.290, still 0.174 too high. After the redesign it is 4.143 against 3.386, still 0.757 too high. The corrected lift is 0.680 against a true lift of 0.096, so weighting barely moved the PM’s number (0.704 to 0.680). Both intervals miss the truth.
3. **In your own words: 2–3 sentences on the lines that compute the weights.** `p_hat = LogisticRegression().fit(X, r).predict_proba(X)[:, 1] `estimates each user’s probability of answering from only engagement and tickets. `w = 1.0 / p_hat[r]` gives each respondent the inverse of that probability, so users who rarely answer count more.
The last line divides by `np.sum(w)` to make a weighted average.
4. **One sentence: would you use AI for this step again, and why.** Yes, for fitting, weighting, bootstrapping, because it was fast and the mechanism check correctly warned that it wouldn’t recover the truth, but I’d still check it against my known true mean.

### Task 4.2: The slide (5 pts)

The survey shows respondents are happier, not that the redesign worked

What it supports:
* Among users who answered, average satisfaction rose from 3.60 to 4.30, a gap of 0.70 points (95% CI 0.683 to 0.725).

What it does not support

* That all users are 0.70 points happier. Response fell from 27% to 15%, and the people still answering lean toward the satisfied. The interval is narrow, but it only describes responders.
* A fix after the fact. Reweighting by engagement and support tickets moved the gap only to 0.68, because the bias comes from satisfaction itself. Surveying more users won’t help either: more responses just give a more precise estimate of the wrong number.

The one change
* Each period, randomly select a fixed set of users (say 5,000) and follow up with non-responders until most have answered. Random selection, not a larger pile of responses, is what makes the number trustworthy.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*

*Version 1 prompt:*

*Reply:*

*Version 2 prompt:*

*Reply:*

##\** See above for AI transcript \**

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.